# Date And Time Helper Agent

In this notebook, I build a small agent that answers questions about dates, using the Hugging Face `smolagents` library.

Like the earlier calculator/converter and text stats/formatter agents, this one does not guess. It works out date differences and reformats dates by calling tools built on Python's own `datetime` module, instead of asking the language model to do date arithmetic in its head.

In this notebook, I will learn how to:

- Write a plain Python function that counts the days between two dates
- Write a plain Python function that reformats a date into a different style
- Turn each one into an agent tool with the `@tool` decorator
- Reject an invalid date or an unknown style inside a tool instead of guessing
- Give an agent both tools and watch it choose, or chain, the right one

Everything here stays small on purpose, so the whole idea fits in one sitting.

## 1. Importing Libraries and Creating the Model

First I import the pieces I need from `smolagents`, plus `datetime` from the standard library for the actual date arithmetic.

`CodeAgent` is the agent that writes and runs Python code to solve a task. `tool` is the decorator I use to turn a plain function into something an agent can call. `InferenceClientModel` is the language model, which runs on the Hugging Face Inference API rather than on my own machine.

In [ ]:
from smolagents import CodeAgent, InferenceClientModel, tool
from datetime import datetime

model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct"
)

## 2. Writing a Basic Date Difference Function

Before I build a tool, I write the date arithmetic as an ordinary Python function.

It takes two dates as plain strings in `YYYY-MM-DD` format and returns the number of days between them. Keeping the logic in a plain function first means I can test it on its own, without an agent or a language model anywhere near it.

In [ ]:
def days_between(start: str, end: str) -> int:
    """Returns the number of days between two ISO dates."""
    start_date = datetime.strptime(start, "%Y-%m-%d")
    end_date = datetime.strptime(end, "%Y-%m-%d")
    return (end_date - start_date).days

## 3. Testing the Date Difference Function

I try the function on a few dates I can check by hand before trusting it with anything else.

In [ ]:
print(days_between("2026-01-01", "2026-01-31"))
print(days_between("2026-03-01", "2026-03-01"))
print(days_between("2026-06-15", "2026-01-01"))

## 4. Handling an Invalid Date

One thing can go wrong with this function: passing a date that is not in `YYYY-MM-DD` format. Right now that raises Python's built-in `ValueError`, but the message does not say which format is expected.

I want the error to say exactly which format is allowed, because this message is exactly what the tool will hand back to the agent later.

In [ ]:
def days_between(start: str, end: str) -> int:
    """Returns the number of days between two ISO dates."""
    try:
        start_date = datetime.strptime(start, "%Y-%m-%d")
    except ValueError:
        raise ValueError(f"Invalid date: {start}. Use YYYY-MM-DD.")
    try:
        end_date = datetime.strptime(end, "%Y-%m-%d")
    except ValueError:
        raise ValueError(f"Invalid date: {end}. Use YYYY-MM-DD.")
    return (end_date - start_date).days

## 5. Testing the Error Handling

I check the failure case directly, catching the error myself so the notebook keeps running and I can read the message that would reach the agent.

In [ ]:
try:
    days_between("2026-13-01", "2026-01-01")
except ValueError as error:
    print(error)

## 6. Turning the Date Difference Function Into a Tool

The function works, but an agent cannot call a plain Python function. It needs a tool.

The `@tool` decorator does the conversion. `smolagents` reads the docstring to build the description the model sees, so I describe the date format carefully.

In [ ]:
@tool
def days_between_tool(start: str, end: str) -> str:
    """
    Returns the number of days between two dates.

    Args:
        start (str): The first date, in YYYY-MM-DD format.
        end (str): The second date, in YYYY-MM-DD format.
    """
    try:
        result = days_between(start, end)
    except ValueError as error:
        return str(error)
    return str(result)

## 7. Testing the Tool on Its Own

Before handing the tool to an agent, I call it directly, the same way the agent would. If something is wrong here, I know the mistake is in my code and not in how the model is using it.

In [ ]:
print(days_between_tool("2026-01-01", "2026-01-31"))
print(days_between_tool("2026-13-01", "2026-01-01"))

## 8. Creating an Agent With the Date Difference Tool

Now I give the tool to a `CodeAgent`. The agent reads the question, decides it needs to work out a gap between two dates, writes a line of Python that calls `days_between_tool`, and turns the result into a sentence.

In [ ]:
days_agent = CodeAgent(
    tools=[days_between_tool],
    model=model
)

## 9. Asking the Agent a Date Difference Question

I ask a question in plain English rather than handing over the two dates directly, so the agent has to work out what to pass to the tool.

In [ ]:
days_agent.run(
    "How many days are there between 2026-01-01 and 2026-03-15?"
)

## 10. Adding a Simple Date Formatter Function

A day count only gets me so far, so I write a second, unrelated function: a date formatter. It supports a small, fixed set of styles, `iso`, `us` and `long`, which is enough to show the idea without turning this into a full date library.

In [ ]:
def format_date(date: str, style: str) -> str:
    """Reformats an ISO date string into a different style."""
    try:
        parsed = datetime.strptime(date, "%Y-%m-%d")
    except ValueError:
        raise ValueError(f"Invalid date: {date}. Use YYYY-MM-DD.")
    if style == "iso":
        return parsed.strftime("%Y-%m-%d")
    if style == "us":
        return parsed.strftime("%m/%d/%Y")
    if style == "long":
        return parsed.strftime("%B %d, %Y")
    raise ValueError(f"Unknown style: {style}. Use iso, us or long.")

## 11. Testing the Formatter Function

Same habit as before: I try it on a date I can check by eye, before it goes anywhere near a tool or an agent.

In [ ]:
print(format_date("2026-09-15", "iso"))
print(format_date("2026-09-15", "us"))
print(format_date("2026-09-15", "long"))

## 12. Testing the Formatter's Error Handling

I check two kinds of failure: an unrecognised style, and a date that is not in `YYYY-MM-DD` format. Both should fail with a message that says exactly what went wrong.

In [ ]:
try:
    format_date("2026-09-15", "short")
except ValueError as error:
    print(error)

try:
    format_date("15-09-2026", "iso")
except ValueError as error:
    print(error)

## 13. Turning the Formatter Into a Tool

Same pattern as the date difference tool: wrap the function in `@tool`, and write a docstring that spells out exactly which style names it understands.

In [ ]:
@tool
def format_date_tool(date: str, style: str) -> str:
    """
    Reformats a date into a different style.

    Args:
        date (str): The date to reformat, in YYYY-MM-DD format.
        style (str): One of "iso", "us" or "long".
    """
    try:
        result = format_date(date, style)
    except ValueError as error:
        return str(error)
    return result

## 14. Testing the Formatter Tool

I run it directly one more time before trusting an agent with it.

In [ ]:
print(format_date_tool("2026-09-15", "long"))
print(format_date_tool("2026-09-15", "short"))

## 15. Combining Both Tools Into One Agent

Now I build one agent with both tools. I do not tell it which one to use for which question; it reads both tool descriptions and decides for itself.

In [ ]:
assistant_agent = CodeAgent(
    tools=[days_between_tool, format_date_tool],
    model=model
)

## 16. Asking a Question That Needs the Date Difference Tool

This question only involves the gap between two dates, so I expect the agent to reach for `days_between_tool` and leave the formatter alone.

In [ ]:
assistant_agent.run(
    "How many days are between 2026-05-01 and 2026-05-20?"
)

## 17. Asking a Question That Needs the Formatter

This time there is nothing to count, just a date to reformat, so a well working agent should pick `format_date_tool` instead.

In [ ]:
assistant_agent.run(
    "Write the date 2026-09-15 out in full, like 'September 15, 2026'."
)

## 18. Asking a Question That Needs Both Tools

This question cannot be answered with one tool call. The agent has to work out a day count and reformat a date, so I can see it chain two tool calls together to reach one answer.

In [ ]:
assistant_agent.run(
    "How many days are between 2026-01-01 and 2026-12-25, and what is "
    "2026-12-25 written in US format?"
)

## 19. Limitations of This Simple Agent

This agent is reliable for the exact styles and format I wrote, but it cannot do anything outside that fixed list.

If I ask it for the weekday a date falls on, there is no style for that. The call below fails cleanly with the error message I wrote, which is exactly the point: a missing style should say so, not quietly return a made up answer. Adding a weekday style later just means writing one more `if` branch and one more test, not redesigning the tool.

In [ ]:
print(format_date_tool("2026-09-15", "weekday"))

## 20. Conclusion and Next Steps

In this notebook I built an agent that works out date differences and reformats dates with exact tools instead of guessing.

What I learned:

- Wrapping date arithmetic in a tool makes the answer exact, because the calculation happens in real Python code, not in the language model's head.
- A tool's docstring is not documentation for me, it is the only instruction the model gets, so the allowed format and styles have to be spelled out in it.
- Testing the plain function, then the error case, then the tool, then the agent, catches a mistake at the earliest point where it is still cheap to fix.
- Giving an agent two tools and letting it choose, or chain, between them works the same way here as it did for text and for units, once each tool fails cleanly on its own.

Next, I want to try an agent that combines a tool like this with a retrieval tool, so it can look something up and then do exact arithmetic on what it finds, instead of treating the two skills as separate notebooks.